# Notebook 01 — EDA & Preprocessing — LIAR Dataset

**Objectif** : Explorer le LIAR Dataset, comprendre la distribution des labels et des métadonnées, puis nettoyer et préparer les données pour la modélisation.

**Pipeline :**
1. Chargement des données brutes
2. EDA complète (textes, métadonnées, compteurs, valeurs manquantes)
3. Preprocessing (nettoyage texte, encodage labels)
4. Export vers `02_stg/`

## 0. Installation des dépendances

In [ ]:
!pip install nltk pandas numpy matplotlib seaborn wordcloud --quiet

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)

print('Dépendances et ressources NLTK prêtes.')

## 0.1 Imports globaux

In [ ]:
import os
import re
import string
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

try:
    from wordcloud import WordCloud
    WORDCLOUD_AVAILABLE = True
except ImportError:
    WORDCLOUD_AVAILABLE = False
    print('wordcloud non disponible — utilisation de barplots à la place.')

# Style global
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 13

# Chemins relatifs à la racine du projet
ROOT = Path('..').resolve()
RAW_DIR = ROOT / 'LIAR_DATA_SET' / '01_raw'
STG_DIR = ROOT / 'LIAR_DATA_SET' / '02_stg'
STG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Racine projet : {ROOT}')
print(f'Données brutes : {RAW_DIR}')
print(f'Staging       : {STG_DIR}')

---
# ÉTAPE 1 — Chargement des données brutes

In [ ]:
# Noms des 14 colonnes du LIAR Dataset
COLUMNS = [
    'id', 'label', 'statement', 'subject', 'speaker', 'job_title',
    'state', 'party', 'barely_true_counts', 'false_counts',
    'half_true_counts', 'mostly_true_counts', 'pants_fire_counts', 'context'
]

# Chargement des trois splits
train = pd.read_csv(RAW_DIR / 'train.tsv', sep='\t', header=None, names=COLUMNS)
test  = pd.read_csv(RAW_DIR / 'test.tsv',  sep='\t', header=None, names=COLUMNS)
valid = pd.read_csv(RAW_DIR / 'valid.tsv', sep='\t', header=None, names=COLUMNS)

splits = {'train': train, 'test': test, 'valid': valid}

print('=== Shapes ===' )
for name, df in splits.items():
    print(f'  {name:6s} : {df.shape}')

In [ ]:
# Types de colonnes
print('=== Types (train) ===')
print(train.dtypes)

In [ ]:
# Aperçu des 5 premières lignes de chaque split
for name, df in splits.items():
    print(f'\n=== {name.upper()} — 5 premières lignes ===')
    display(df.head())

---
# ÉTAPE 2 — EDA complète

## 2.1 Distribution des labels

In [ ]:
# Ordre cohérent des labels (du plus faux au plus vrai)
LABEL_ORDER = ['pants-fire', 'false', 'barely-true', 'half-true', 'mostly-true', 'true']
LABEL_COLORS = ['#d62728', '#ff7f0e', '#ffbb78', '#aec7e8', '#1f77b4', '#2ca02c']

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for ax, (name, df) in zip(axes, splits.items()):
    counts = df['label'].value_counts().reindex(LABEL_ORDER, fill_value=0)
    bars = ax.bar(counts.index, counts.values, color=LABEL_COLORS, edgecolor='white')
    ax.set_title(f'Distribution des labels — {name.capitalize()}')
    ax.set_xlabel('Label')
    ax.set_ylabel('Nombre de déclarations')
    ax.tick_params(axis='x', rotation=30)
    # Afficher les valeurs au-dessus des barres
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)

plt.suptitle('Distribution des 6 labels de véracité par split', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Ratio de déséquilibre : classe majoritaire / classe minoritaire
print('=== Ratio de déséquilibre (majoritaire / minoritaire) ===')
for name, df in splits.items():
    counts = df['label'].value_counts()
    ratio = counts.max() / counts.min()
    print(f'  {name:6s} : max={counts.max()} ({counts.idxmax()}), '
          f'min={counts.min()} ({counts.idxmin()}), ratio={ratio:.2f}')

## 2.2 Analyse des textes

In [ ]:
# Calcul des longueurs (mots et caractères) sur le split train
train['word_count'] = train['statement'].dropna().apply(lambda x: len(str(x).split()))
train['char_count'] = train['statement'].dropna().apply(lambda x: len(str(x)))

print('=== Statistiques de longueur des statements (train) ===')
print(train[['word_count', 'char_count']].describe().round(1))

In [ ]:
# Histogrammes de longueur
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train['word_count'].dropna(), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution du nombre de mots par statement (train)')
axes[0].set_xlabel('Nombre de mots')
axes[0].set_ylabel('Fréquence')

axes[1].hist(train['char_count'].dropna(), bins=50, color='darkorange', edgecolor='white')
axes[1].set_title('Distribution du nombre de caractères par statement (train)')
axes[1].set_xlabel('Nombre de caractères')
axes[1].set_ylabel('Fréquence')

plt.tight_layout()
plt.show()

In [ ]:
# Longueur moyenne des statements par label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(
    axes,
    ['word_count', 'char_count'],
    ['Mots', 'Caractères']
):
    means = (train.groupby('label')[col]
             .mean()
             .reindex(LABEL_ORDER)
             .dropna())
    ax.bar(means.index, means.values, color=LABEL_COLORS, edgecolor='white')
    ax.set_title(f'Longueur moyenne ({title}) par label')
    ax.set_xlabel('Label')
    ax.set_ylabel(f'Moyenne {title.lower()}')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Top 20 mots les plus fréquents (après suppression des stopwords)
stop_words = set(stopwords.words('english'))

def tokenize_no_stop(text):
    """Tokenise un texte et retire les stopwords et la ponctuation."""
    tokens = re.sub(r'[^a-zA-Z\s]', '', str(text).lower()).split()
    return [t for t in tokens if t not in stop_words and len(t) > 2]

# Concaténation de tous les tokens du train
all_tokens = []
for stmt in train['statement'].dropna():
    all_tokens.extend(tokenize_no_stop(stmt))

top20 = Counter(all_tokens).most_common(20)
words_top20, counts_top20 = zip(*top20)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(list(reversed(words_top20)), list(reversed(counts_top20)), color='steelblue')
ax.set_title('Top 20 mots les plus fréquents (train, sans stopwords)')
ax.set_xlabel('Fréquence')
plt.tight_layout()
plt.show()

In [ ]:
# Top 20 mots par label extrême : pants-fire vs true
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, target_label, color in zip(
    axes,
    ['pants-fire', 'true'],
    ['#d62728', '#2ca02c']
):
    subset = train[train['label'] == target_label]['statement'].dropna()
    tokens = []
    for stmt in subset:
        tokens.extend(tokenize_no_stop(stmt))
    top = Counter(tokens).most_common(20)
    if top:
        ws, cs = zip(*top)
        ax.barh(list(reversed(ws)), list(reversed(cs)), color=color)
    ax.set_title(f'Top 20 mots — label "{target_label}" (train)')
    ax.set_xlabel('Fréquence')

plt.suptitle('Comparaison des mots fréquents : pants-fire vs true', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Wordclouds (ou barplots si wordcloud non disponible)
if WORDCLOUD_AVAILABLE:
    wc_colors = {
        'pants-fire': 'Reds', 'false': 'Oranges', 'barely-true': 'YlOrBr',
        'half-true': 'Blues', 'mostly-true': 'GnBu', 'true': 'Greens'
    }
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for ax, lbl in zip(axes, LABEL_ORDER):
        subset = train[train['label'] == lbl]['statement'].dropna()
        text = ' '.join([' '.join(tokenize_no_stop(s)) for s in subset])
        if text.strip():
            wc = WordCloud(
                width=600, height=300,
                background_color='white',
                colormap=wc_colors.get(lbl, 'viridis'),
                max_words=80
            ).generate(text)
            ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(f'Label : {lbl}', fontsize=12, fontweight='bold')

    plt.suptitle('Wordclouds par label de véracité (train)', fontsize=15, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    # Fallback : barplots top-10 mots pour chaque label
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for ax, lbl in zip(axes, LABEL_ORDER):
        subset = train[train['label'] == lbl]['statement'].dropna()
        tokens = []
        for s in subset:
            tokens.extend(tokenize_no_stop(s))
        top10 = Counter(tokens).most_common(10)
        if top10:
            ws, cs = zip(*top10)
            ax.barh(list(reversed(ws)), list(reversed(cs)))
        ax.set_title(f'Top 10 mots — {lbl}')
    plt.tight_layout()
    plt.show()

## 2.3 Analyse des métadonnées

In [ ]:
# Top 20 speakers les plus représentés (train)
top_speakers = train['speaker'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_speakers.index[::-1], top_speakers.values[::-1], color='steelblue')
ax.set_title('Top 20 speakers les plus représentés (train)')
ax.set_xlabel('Nombre de déclarations')
ax.set_ylabel('Speaker')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des labels par parti politique
# Regroupement : Republican, Democrat, autres
def simplify_party(p):
    p = str(p).strip().lower()
    if 'republican' in p:
        return 'Republican'
    elif 'democrat' in p:
        return 'Democrat'
    else:
        return 'Autre'

train['party_group'] = train['party'].apply(simplify_party)

party_label = train.groupby(['party_group', 'label']).size().reset_index(name='count')

fig, ax = plt.subplots(figsize=(12, 5))
party_order = ['Democrat', 'Republican', 'Autre']
pivot = party_label.pivot(index='party_group', columns='label', values='count').fillna(0)
pivot = pivot.reindex(party_order).fillna(0)
pivot[LABEL_ORDER].plot(kind='bar', ax=ax, color=LABEL_COLORS, edgecolor='white')
ax.set_title('Distribution des labels par parti politique (train)')
ax.set_xlabel('Parti politique')
ax.set_ylabel('Nombre de déclarations')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='Label', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap : croisement parti politique × label
pivot_pct = pivot[LABEL_ORDER].div(pivot[LABEL_ORDER].sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    pivot_pct,
    annot=True, fmt='.1f', cmap='RdYlGn',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': '% dans le groupe parti'}
)
ax.set_title('Heatmap : % de chaque label par parti politique (train)')
ax.set_xlabel('Label')
ax.set_ylabel('Parti')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des sujets (subject) les plus fréquents
# Un statement peut avoir plusieurs sujets séparés par une virgule
all_subjects = []
for s in train['subject'].dropna():
    all_subjects.extend([x.strip() for x in str(s).split(',')])

top_subjects = Counter(all_subjects).most_common(20)
sub_words, sub_counts = zip(*top_subjects)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(list(reversed(sub_words)), list(reversed(sub_counts)), color='darkorange')
ax.set_title('Top 20 sujets les plus fréquents (train)')
ax.set_xlabel('Fréquence')
ax.set_ylabel('Sujet')
plt.tight_layout()
plt.show()

## 2.4 Analyse des compteurs historiques

In [ ]:
# Colonnes de compteurs historiques
COUNT_COLS = ['barely_true_counts', 'false_counts', 'half_true_counts',
              'mostly_true_counts', 'pants_fire_counts']

# Conversion en numérique (peuvent contenir des NaN)
for col in COUNT_COLS:
    train[col] = pd.to_numeric(train[col], errors='coerce')

# Distribution des compteurs
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, col in zip(axes, COUNT_COLS):
    data = train[col].dropna()
    ax.hist(data, bins=40, color='steelblue', edgecolor='white')
    ax.set_title(col.replace('_counts', '').replace('_', '-'), fontsize=10)
    ax.set_xlabel('Comptage')
    ax.set_ylabel('Fréquence')

plt.suptitle('Distribution des compteurs historiques de véracité (train)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Moyennes des compteurs par label
count_by_label = train.groupby('label')[COUNT_COLS].mean().reindex(LABEL_ORDER)

fig, ax = plt.subplots(figsize=(12, 5))
count_by_label.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='white')
ax.set_title('Moyennes des compteurs historiques par label (train)')
ax.set_xlabel('Label')
ax.set_ylabel('Moyenne du compteur')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Compteur', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Corrélation entre compteurs et label (encodé numériquement pour le calcul)
label_num_map = {l: i for i, l in enumerate(LABEL_ORDER)}
train['label_num'] = train['label'].map(label_num_map)

corr_data = train[COUNT_COLS + ['label_num']].dropna()
corr_matrix = corr_data.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.5, ax=ax
)
ax.set_title('Corrélation entre compteurs historiques et label numérique')
plt.tight_layout()
plt.show()

## 2.5 Détection des valeurs manquantes

In [ ]:
# Pourcentage de nulls par colonne pour chaque split
print('=== Pourcentage de valeurs nulles par colonne ===')
null_summary = pd.DataFrame({
    name: df[COLUMNS].isnull().mean() * 100
    for name, df in splits.items()
})
print(null_summary.round(2).to_string())

In [ ]:
# Heatmap des valeurs nulles (train)
null_matrix = train[COLUMNS].isnull().astype(int)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    null_matrix.T,
    cmap='YlOrRd', cbar_kws={'label': '1 = valeur manquante'},
    linewidths=0, ax=ax
)
ax.set_title('Carte des valeurs manquantes — train')
ax.set_xlabel('Index de la ligne')
ax.set_ylabel('Colonne')
plt.tight_layout()
plt.show()

---
# ÉTAPE 3 — Preprocessing

## 3.1 Nettoyage du texte

In [ ]:
# Initialisation du lemmatizer et des stopwords
lemmatizer = WordNetLemmatizer()
stop_words_set = set(stopwords.words('english'))

def clean_text(text):
    """Pipeline de nettoyage complet d'un statement.

    Étapes :
    1. Conversion en string et lowercase
    2. Suppression de la ponctuation et des caractères spéciaux
    3. Suppression des stopwords
    4. Lemmatisation
    """
    # 1. Lowercase
    text = str(text).lower()
    # 2. Suppression de la ponctuation et caractères non-alphabétiques
    text = re.sub(r'[^a-z\s]', '', text)
    # 3. Tokenisation
    tokens = text.split()
    # 4. Suppression des stopwords et mots trop courts
    tokens = [t for t in tokens if t not in stop_words_set and len(t) > 2]
    # 5. Lemmatisation
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

print('Fonction clean_text définie. Test rapide :')
test_stmt = "The President said he would never raise taxes on the middle-class families!"
print(f'  Original : {test_stmt}')
print(f'  Nettoyé  : {clean_text(test_stmt)}')

In [ ]:
# Application du nettoyage sur les 3 splits
# Note : les compteurs sont aussi convertis en numérique ici
for name, df in splits.items():
    # Nettoyage du texte
    df['clean_statement'] = df['statement'].apply(clean_text)
    # Conversion des compteurs en numérique
    for col in COUNT_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f'{name:6s} : clean_statement créée. Exemple : {df["clean_statement"].iloc[0][:80]}...')

## 3.2 Mapping des labels

In [ ]:
# label_binary : 0 = fake, 1 = real
BINARY_MAP = {
    'pants-fire': 0, 'false': 0, 'barely-true': 0,  # fake
    'half-true': 1, 'mostly-true': 1, 'true': 1      # real
}

# label_3class : 0 = faux, 1 = mixte, 2 = vrai
TRICLASS_MAP = {
    'pants-fire': 0, 'false': 0,         # faux
    'barely-true': 1, 'half-true': 1,    # mixte
    'mostly-true': 2, 'true': 2          # vrai
}

for name, df in splits.items():
    df['label_binary']  = df['label'].map(BINARY_MAP)
    df['label_3class']  = df['label'].map(TRICLASS_MAP)

print('Mappings des labels appliqués. Distribution label_binary (train) :')
print(train['label_binary'].value_counts().rename({0: 'fake (0)', 1: 'real (1)'}).to_string())

print('\nDistribution label_3class (train) :')
print(train['label_3class'].value_counts().rename({0: 'faux (0)', 1: 'mixte (1)', 2: 'vrai (2)'}).to_string())

## 3.3 Encodage des métadonnées

In [ ]:
# Encodage du parti politique : Republican=0, Democrat=1, autres=2
PARTY_ENCODE = {'Republican': 0, 'Democrat': 1, 'Autre': 2}

for name, df in splits.items():
    df['party_group'] = df['party'].apply(simplify_party)
    df['party_encoded'] = df['party_group'].map(PARTY_ENCODE)

print('Encodage du parti politique :')
print(train[['party', 'party_group', 'party_encoded']].drop_duplicates().head(10).to_string(index=False))

## 3.4 Vérification finale

In [ ]:
# Shape avant/après (les colonnes ajoutées augmentent la largeur)
ORIGINAL_COLS = COLUMNS
ADDED_COLS = ['clean_statement', 'label_binary', 'label_3class', 'party_group', 'party_encoded']

print('=== Shape avant / après preprocessing ===')
for name, df in splits.items():
    print(f'  {name:6s} : {len(ORIGINAL_COLS)} colonnes originales → {df.shape[1]} colonnes totales ({df.shape[0]} lignes)')

In [ ]:
# Vérification des nulls dans clean_statement
print('=== Valeurs nulles dans clean_statement ===')
for name, df in splits.items():
    n_null = df['clean_statement'].isnull().sum()
    n_empty = (df['clean_statement'] == '').sum()
    print(f'  {name:6s} : {n_null} null, {n_empty} chaînes vides')

In [ ]:
# 5 exemples côte à côte : statement original vs clean_statement
print('=== 5 exemples : statement original vs clean_statement (train) ===')
examples = train[['statement', 'clean_statement', 'label', 'label_binary', 'label_3class']].head(5)
pd.set_option('display.max_colwidth', 80)
display(examples)

---
# ÉTAPE 4 — Export vers 02_stg/

In [ ]:
# Colonnes finales à exporter
EXPORT_COLS = COLUMNS + ADDED_COLS

export_map = {
    'train': train,
    'test':  test,
    'valid': valid
}

exported_paths = []

for name, df in export_map.items():
    # Sélection des colonnes disponibles (certaines ajoutées seulement sur train au début)
    cols_to_export = [c for c in EXPORT_COLS if c in df.columns]
    out_path = STG_DIR / f'{name}_clean.csv'
    df[cols_to_export].to_csv(out_path, index=False)
    exported_paths.append(out_path)
    print(f'Export : {out_path}  →  shape {df[cols_to_export].shape}')

print('\nExport terminé.')

In [ ]:
# Résumé final
print('=' * 60)
print('RÉSUMÉ FINAL — Fichiers exportés dans 02_stg/')
print('=' * 60)

for path in exported_paths:
    df_check = pd.read_csv(path)
    print(f'\nFichier : {path.name}')
    print(f'  Chemin : {path}')
    print(f'  Shape  : {df_check.shape}')
    print(f'  Colonnes : {list(df_check.columns)}')
    display(df_check.head(3))